# Notebook 03d — Ensembling & Final Evaluation

**Question:** Can combining models beat the best individual? And what's the final test score?

We build Soft Voting and Stacking ensembles from tuned models,
then evaluate the best ML model on the held-out test set (one shot, no re-tuning).

**Inputs:** `artifacts/` from nb02, `artifacts/ml_best_params.pkl` from nb03c
**Outputs:** `results/ml_ensemble.csv`, `results/ml_final_test.csv`, `models/` (saved models)


In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, os, joblib
import matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier, StackingClassifier
import xgboost as xgb
import lightgbm as lgb

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")
SEED = 42; np.random.seed(SEED)
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)
print("Imports OK")


## 0. Load Data & Tuned Params

In [ ]:
# All splits
X_train_3c = joblib.load('../artifacts/X_train_3c.pkl').astype(np.float32)
X_val_3c   = joblib.load('../artifacts/X_val_3c.pkl').astype(np.float32)
X_test_3c  = joblib.load('../artifacts/X_test_3c.pkl').astype(np.float32)
y_train_3c = joblib.load('../artifacts/y_train_3c.pkl')
y_val_3c   = joblib.load('../artifacts/y_val_3c.pkl')
y_test_3c  = joblib.load('../artifacts/y_test_3c.pkl')

X_train_bin = joblib.load('../artifacts/X_train_bin.pkl').astype(np.float32)
X_val_bin   = joblib.load('../artifacts/X_val_bin.pkl').astype(np.float32)
X_test_bin  = joblib.load('../artifacts/X_test_bin.pkl').astype(np.float32)
y_train_bin = joblib.load('../artifacts/y_train_bin.pkl')
y_val_bin   = joblib.load('../artifacts/y_val_bin.pkl')
y_test_bin  = joblib.load('../artifacts/y_test_bin.pkl')

le_3class = joblib.load('../artifacts/label_encoder_3class.pkl')
feature_names = joblib.load('../artifacts/feature_names.pkl')
best_params = joblib.load('../artifacts/ml_best_params.pkl')

# For CV
X_3c = np.vstack([X_train_3c, X_val_3c]); y_3c = np.concatenate([y_train_3c, y_val_3c])
X_bin = np.vstack([X_train_bin, X_val_bin]); y_bin = np.concatenate([y_train_bin, y_val_bin])

print(f'3-class: CV={X_3c.shape}, Test={X_test_3c.shape}')
print(f'Binary:  CV={X_bin.shape}, Test={X_test_bin.shape}')
print(f'Best params loaded: {list(best_params.keys())}')


In [ ]:
SCOREBOARD = []
def log_exp(model, strategy, cv_mean, cv_std, n_train, notes=""):
    SCOREBOARD.append(dict(phase="ensemble", model=model, strategy=strategy,
        n_features=len(feature_names), cv_f1_mean=round(cv_mean,4),
        cv_f1_std=round(cv_std,4), n_train=n_train, notes=notes))
    print(f"  {model:22s} | {strategy:8s} | F1={cv_mean:.4f} ± {cv_std:.4f}")

def run_cv(model, X, y):
    cv = StratifiedKFold(5, shuffle=True, random_state=SEED)
    s = cross_val_score(model, X, y, cv=cv, scoring="f1_macro", n_jobs=-1)
    return s.mean(), s.std()

# Model builders
def xgb_3c():
    p = best_params['xgb_3c']
    return xgb.XGBClassifier(max_depth=p["max_depth"], learning_rate=p["learning_rate"],
        n_estimators=p["n_estimators"], subsample=p["subsample"],
        colsample_bytree=p["colsample_bytree"], min_child_weight=p["min_child_weight"],
        reg_lambda=p["reg_lambda"], random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="mlogloss")

def lgb_3c():
    p = best_params['lgb_3c']
    return lgb.LGBMClassifier(num_leaves=p["num_leaves"], learning_rate=p["learning_rate"],
        n_estimators=p["n_estimators"], min_child_samples=p["min_child_samples"],
        feature_fraction=p["feature_fraction"], bagging_fraction=p["bagging_fraction"],
        bagging_freq=p["bagging_freq"], class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1)

def xgb_bin():
    p = best_params['xgb_bin']
    return xgb.XGBClassifier(max_depth=p["max_depth"], learning_rate=p["learning_rate"],
        n_estimators=p["n_estimators"], subsample=p["subsample"],
        colsample_bytree=p["colsample_bytree"], min_child_weight=p["min_child_weight"],
        reg_lambda=p["reg_lambda"], random_state=SEED, n_jobs=-1, verbosity=0, eval_metric="logloss")

def lgb_bin():
    p = best_params['lgb_bin']
    return lgb.LGBMClassifier(num_leaves=p["num_leaves"], learning_rate=p["learning_rate"],
        n_estimators=p["n_estimators"], min_child_samples=p["min_child_samples"],
        feature_fraction=p["feature_fraction"], bagging_fraction=p["bagging_fraction"],
        bagging_freq=p["bagging_freq"], class_weight="balanced",
        random_state=SEED, n_jobs=-1, verbose=-1)

rf = lambda: RandomForestClassifier(n_estimators=200, class_weight="balanced", random_state=SEED, n_jobs=-1)
lr = lambda: LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED)


## 1. Ensembles — 3-class

In [ ]:
print("=== 3-class Ensembles ===")

# Soft Voting
vote_3c = VotingClassifier(
    estimators=[("xgb", xgb_3c()), ("lgb", lgb_3c()), ("rf", rf())],
    voting="soft", n_jobs=-1)
m, s = run_cv(vote_3c, X_3c, y_3c)
log_exp("SoftVoting", "3class", m, s, len(X_3c), "xgb+lgb+rf")

# Stacking
stack_3c = StackingClassifier(
    estimators=[("xgb", xgb_3c()), ("lgb", lgb_3c()), ("rf", rf())],
    final_estimator=lr(), cv=3, n_jobs=-1)
m, s = run_cv(stack_3c, X_3c, y_3c)
log_exp("Stacking", "3class", m, s, len(X_3c), "xgb+lgb+rf→lr")


## 2. Ensembles — Binary

In [ ]:
print("=== Binary Ensembles ===")

vote_bin = VotingClassifier(
    estimators=[("xgb", xgb_bin()), ("lgb", lgb_bin()), ("rf", rf())],
    voting="soft", n_jobs=-1)
m, s = run_cv(vote_bin, X_bin, y_bin)
log_exp("SoftVoting", "binary", m, s, len(X_bin), "xgb+lgb+rf")

stack_bin = StackingClassifier(
    estimators=[("xgb", xgb_bin()), ("lgb", lgb_bin()), ("rf", rf())],
    final_estimator=lr(), cv=3, n_jobs=-1)
m, s = run_cv(stack_bin, X_bin, y_bin)
log_exp("Stacking", "binary", m, s, len(X_bin), "xgb+lgb+rf→lr")


## 3. Final Test Set Evaluation

⚠️ **One shot. No re-tuning after this.**

We train the best models on train+val, then evaluate on the held-out test set.


In [ ]:
print("=" * 60)
print("FINAL TEST SET EVALUATION")
print("=" * 60)

test_results = []

# ── 3-class: best individual + best ensemble ──
print("\n--- 3-class ---")

# Train on full train+val
xgb_3c_final = xgb_3c()
xgb_3c_final.fit(X_3c, y_3c)
y_pred = xgb_3c_final.predict(X_test_3c)
f1_m = f1_score(y_test_3c, y_pred, average='macro')
print(f"\nXGBoost-tuned: Macro F1 = {f1_m:.4f}")
print(classification_report(y_test_3c, y_pred, target_names=le_3class.classes_))
test_results.append(dict(model="XGBoost-tuned", strategy="3class", test_f1=round(f1_m, 4)))

# Confusion matrix
cm = confusion_matrix(y_test_3c, y_pred, normalize='true')
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=le_3class.classes_, yticklabels=le_3class.classes_, ax=ax)
ax.set_title('XGBoost-tuned — 3-class Test Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('results/cm_xgb_3class_test.png', bbox_inches='tight')
plt.show()

# Save model
joblib.dump(xgb_3c_final, '../models/xgb_tuned_3class.pkl')

# ── Binary ──
print("\n--- Binary ---")
stack_bin_final = StackingClassifier(
    estimators=[("xgb", xgb_bin()), ("lgb", lgb_bin()), ("rf", rf())],
    final_estimator=lr(), cv=3, n_jobs=-1)
stack_bin_final.fit(X_bin, y_bin)
y_pred_bin = stack_bin_final.predict(X_test_bin)
f1_b = f1_score(y_test_bin, y_pred_bin, average='macro')
print(f"\nStacking: Macro F1 = {f1_b:.4f}")
print(classification_report(y_test_bin, y_pred_bin, target_names=['failed', 'success']))
test_results.append(dict(model="Stacking", strategy="binary", test_f1=round(f1_b, 4)))

cm_b = confusion_matrix(y_test_bin, y_pred_bin, normalize='true')
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_b, annot=True, fmt='.2f', cmap='Greens',
            xticklabels=['failed', 'success'], yticklabels=['failed', 'success'], ax=ax)
ax.set_title('Stacking — Binary Test Confusion Matrix')
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.tight_layout()
plt.savefig('results/cm_stacking_binary_test.png', bbox_inches='tight')
plt.show()

joblib.dump(stack_bin_final, '../models/stacking_binary.pkl')


In [ ]:
# Save all results
sb = pd.DataFrame(SCOREBOARD)
sb.to_csv('results/ml_ensemble.csv', index=False)

test_df = pd.DataFrame(test_results)
test_df.to_csv('results/ml_final_test.csv', index=False)

print(f"Saved: results/ml_ensemble.csv ({len(sb)} experiments)")
print(f"Saved: results/ml_final_test.csv")
print(f"Saved: models/xgb_tuned_3class.pkl")
print(f"Saved: models/stacking_binary.pkl")
print()
print("Ensemble CV scores:")
print(sb.to_string(index=False))
print()
print("Final test scores:")
print(test_df.to_string(index=False))
